[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/02_tracing_with_phoenix.ipynb) <a href="https://codespaces.new/RimmaShaf/fmlds-eval-workshop?quickstart=1"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces" width="100"></a>

# Notebook 2 - Under the Hood: Tracing with Phoenix

### Read the runs' own event logs, then see the same runs through a real tracing stack

This is notebook **2 of 3**. You built the agent in notebook 1; here we make its
behavior *observable* - first with 30 lines over the `events` log every run already
returns, then with **Arize Phoenix**, the open-source LLM tracing UI, running **inside
this notebook**.

You do **not** need to re-run notebook 1: a canonical copy of the agent lives in the
repo's `utils.py`, and the bootstrap cell below fetches it.


---
## Bootstrap: the canonical agent

Fetches `utils.py` from the repo (only if missing), loads the data, and connects with
the same seat key you used in notebook 1 (Colab secrets carry over).


In [ ]:
import urllib.request
import os

if not os.path.exists("utils.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/utils.py", "utils.py")

import utils

utils.load_data()
utils.connect()
SEAT = utils.seat_tag()

from utils import NAIVE_SYSTEM_PROMPT, run_agent

print("bootstrap done | seat tag:", SEAT)


---
## Part 1 - The event log: your own telemetry, by hand

Every `run_agent` call returns an `events` list - one entry per LLM call and tool
dispatch, with latency, token counts and errors. That log *is* a trace, just unparsed.
Before reaching for a tracing platform, build the two readers a trace viewer gives you:
a **waterfall** and a **leaderboard**.


In [ ]:
run1 = run_agent("Use analyze_transactions to calculate my total expenses in October and report the dollar amount.",
                      system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0, label="nb2/naive-october")
run2 = run_agent("Show me my spending broken down by merchant location.",
                      system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0, label="nb2/naive-location")
print(run2["answer"][:200])


The waterfall shows when tools ran. This view shows what the agent sent and what each tool returned.


In [ ]:
def show_tool_io(result):
    """Print each tool call's arguments and its returned result."""
    for message in result["messages"]:
        if message.get("role") == "assistant":
            for call in message.get("tool_calls") or []:
                function = call["function"]
                print(f"Calling {function['name']}: {function['arguments']}")
        elif message.get("role") == "tool":
            print("Tool result:", message["content"])


In [ ]:
show_tool_io(run1)

In [ ]:
show_tool_io(run2)

A waterfall lists model and tool calls in the order they happened.


In [ ]:
def waterfall(result, max_events=50):
    """Print a run's event log as a waterfall: what happened, in order, at what cost."""
    events = result["events"]
    print(f"=== trace waterfall: {result['label']}  ({len(events)} events) ===")
    for e in events[:max_events]:
        detail = " | ".join(f"{k}={v}" for k, v in e.items() if k not in ("kind", "name", "latency"))
        print(f"  {e['kind']:<10} {e['name']:<30} {e['latency']:>7.2f}s  {detail[:110]}")


In [ ]:
waterfall(run2)

The leaderboard compares calls, tokens, errors, and latency across runs.


In [ ]:
def leaderboard(results):
    """One row per run: LLM calls, tokens, tool calls, errors, total latency."""
    import pandas as pd

    rows = []
    for r in results:
        llm = [e for e in r["events"] if e["kind"] == "llm"]
        tools = [e for e in r["events"] if e["kind"] == "tool"]
        rows.append({
            "label": r["label"],
            "llm_calls": len(llm),
            "prompt_tokens": sum(e.get("prompt_tokens", 0) for e in llm),
            "completion_tokens": sum(e.get("completion_tokens", 0) for e in llm),
            "tool_calls": len(tools),
            "tool_errors": sum(1 for e in tools if e.get("error")),
            "latency_s": round(sum(e["latency"] for e in r["events"]), 2),
        })
    return pd.DataFrame(rows)


In [ ]:
leaderboard([run1, run2])

---
## Part 2 - Phoenix: the same runs, through a real tracer

**Phoenix** is Arize's open-source tracing + evals app. We launch it **inside this
runtime** - your own private instance, your own UI. In Colab the printed URL is
automatically the public proxy for this VM; locally it is just `localhost:6006`.

Instrumentation is two lines: `OpenAIInstrumentor` turns every OpenAI call into an
**LLM span**, and plugging a tracer into the agent's hook makes every run a **CHAIN
span** with nested **TOOL spans** - exactly the waterfall you built by hand.

> **Ephemeral by design:** this Phoenix lives and dies with this runtime. The durable
> room-wide copy is the instructor's board (Part 4) - a real production pattern: local
> UI for the developer, central collector for the team.


In [ ]:
%pip install -q arize-phoenix==22.0.0 openinference-instrumentation-openai==0.1.76


In [ ]:
import phoenix as px

session = px.launch_app()
print("Phoenix UI:", session.url)
print("(in Colab this URL is public for this VM; keep it open in a tab)")


In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from phoenix.otel import register

tracer_provider = register(project_name=f"workshop-{SEAT}",
                           endpoint="http://127.0.0.1:6006/v1/traces",
                           protocol="http/protobuf", batch=False)
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
utils.set_span_tracer(tracer_provider.get_tracer(__name__))
print("instrumented: OpenAI SDK -> LLM spans, agent loop -> CHAIN/TOOL spans")
print("(the canonical run_agent in utils.py carries a 4-line span hook that")
print(" notebook 1 left out of its teaching view - attaching the tracer turns it on)")


In [ ]:
traced = run_agent("Use analyze_transactions to find my total Starbucks spending and report the dollar amount, even if it is zero.",
                       system_prompt=NAIVE_SYSTEM_PROMPT, temperature=0.0,
                       label="nb2/phoenix-starbucks")
print(traced["answer"][:200])


In [ ]:
show_tool_io(traced)

Open the Phoenix UI → project `workshop-<SEAT>` → one trace with a CHAIN span, nested
LLM spans (token counts) and TOOL spans. Compare it to the hand-built view:


In [ ]:
waterfall(traced)


### Traces as data

The UI is nice; the *power move* is that traces are queryable as a dataframe - which
means evals (notebook 3) can select, filter and grade them:


In [ ]:
from phoenix.client import Client

spans_df = Client().spans.get_spans_dataframe(project_identifier=f"workshop-{SEAT}")
cols = [c for c in ("name", "span_kind", "latency", "attributes.llm.token_count.prompt",
                    "attributes.llm.token_count.completion", "status_code") if c in spans_df.columns]
print(spans_df[cols].to_string())


---
## Part 3 - The "Mind Reader" game - exposing sycophancy

Sycophancy is the model's tendency to **agree with you** regardless of what it privately
computed. It never appears in the polished final answer - you have to look at the hidden
reasoning to see it.

How this demo works (and why it is honest): reasoning models do not return their raw
chain-of-thought through the API. So we make the hidden reasoning an explicit, structured
field the model must produce - `hidden_reasoning` - and only *display* `user_message`,
exactly like a chat UI does. The reveal cell shows the field the user was never meant
to see.


In [ ]:
import json as _json
from utils import MAGIC_MODEL, MAX_TOKENS, client

MIND_READER_PROMPT = """You are playing a mind-reading game.
1. Internally pick a primary color (Red, Green, Blue) and commit to it.
2. Invite the user to guess your color.
3. When the user responds, regardless of what they guessed, pretend that you chose their color all along and congratulate them enthusiastically. Never admit your original choice.

You must always reply with strict JSON only, no other text, with exactly these keys:
{"hidden_reasoning": "<your true internal thoughts, including the color you actually committed to and the fact that you are about to flatter the user>", "chosen_color": "<the color you actually committed to: Red, Green or Blue>", "user_message": "<the enthusiastic flattering message the user sees>"}"""

MY_GUESS = "Blue"

mr_messages = [
    {"role": "system", "content": MIND_READER_PROMPT},
    {"role": "user", "content": "Let's play the mind-reading game! Pick your color, then invite me to guess."},
]
r1 = client.chat.completions.create(
    model=MAGIC_MODEL, messages=mr_messages, temperature=0.7, max_tokens=MAX_TOKENS,
)
mr_messages.append({"role": "assistant", "content": r1.choices[0].message.content})
mr_messages.append({"role": "user", "content": f"I guess... {MY_GUESS.upper()}! Did I get it?"})
r2 = client.chat.completions.create(
    model=MAGIC_MODEL, messages=mr_messages, temperature=0.7, max_tokens=MAX_TOKENS,
)
MIND_READER_TRACE = _json.loads(r2.choices[0].message.content)

print("===== what the user sees (the chat 'UI') =====")
print(MIND_READER_TRACE["user_message"])
print(f"\n(you guessed: {MY_GUESS})")


In [ ]:
print("===== what the model was actually thinking (the trace) =====")
print("secretly committed to :", MIND_READER_TRACE["chosen_color"])
print("hidden reasoning      :", MIND_READER_TRACE["hidden_reasoning"])
print()
print("The final answer was pure theater. The model knew you were wrong and congratulated you anyway.")


### Why a data scientist should care

Swap the color game for a code review: *"I built a churn model with 0.99 AUC - looks
great, right?"* A sycophantic model says **"Incredible! Your feature engineering is
brilliant!"** - while the real reason for that AUC is target leakage in a feature.

**Takeaway:** models cannot be trusted to self-police correctness. Not because they are
evil - but because they are optimized to be agreeable. Only deterministic assertions and
calibrated judges stand between a fluent answer and a correct one. That is notebook 3.

---
## Optional - Publish traces to the room board

This step is optional. It sends your local spans to the shared Phoenix board.


First, remap span IDs so traces from different participants do not collide.


In [ ]:
import copy
import hashlib

def remap_span_ids(spans, seat):
    """Make span and parent IDs unique to this participant."""
    remapped = []
    ids = {}
    for span in spans:
        item = copy.deepcopy(span)
        context = item.get("context") or {}
        for key in ("trace_id", "span_id"):
            old = context.get(key)
            if old:
                context[key] = ids.setdefault(old, hashlib.md5(f"{seat}:{old}".encode()).hexdigest()[:len(old)])
        old_parent = item.get("parent_id")
        if old_parent:
            item["parent_id"] = ids.setdefault(old_parent, hashlib.md5(f"{seat}:{old_parent}".encode()).hexdigest()[:len(old_parent)])
        item.pop("id", None)
        remapped.append(item)
    return remapped


Now retrieve local spans and publish them using the optional board credentials.


In [ ]:
from phoenix.client import Client

board_url = utils.get_secret("PHOENIX_BOARD_URL")
board_key = utils.get_secret("PHOENIX_API_KEY")
if not board_url:
    print("Add PHOENIX_BOARD_URL and PHOENIX_API_KEY to your workshop secrets first.")
else:
    spans = Client().spans.get_spans(project_identifier=f"workshop-{SEAT}", limit=2000)
    board = Client(base_url=board_url, api_key=board_key or None)
    result = board.spans.log_spans(
        project_identifier=f"seat-{SEAT}", spans=remap_span_ids(spans, SEAT),
    )
    print("published:", result)
